# 03 - DenseNet121 fine-tune for lung opacity

**Run this notebook on a hosted GPU - Kaggle (sidebar > Accelerator > GPU T4, Internet left off) or Colab
(`Runtime > Change runtime type > T4`) - not on the laptop.** The laptop has 8 GB RAM and no CUDA device; the baseline
run in `02_baseline_logreg.ipynb` spent 93 % of its wall clock just decoding DICOM files, and fine-tuning a CNN on
26,684 images there would take days. Cell 4 reads the host off the filesystem (`RUN_ON`), so one notebook file serves
both platforms. It also recognises a laptop and records `host: local` in `densenet_metrics.json`: that run is a smoke
test of the code, and it must never be quoted as a result.

What this notebook adds over the baselines in notebook 02:

| | model | test AUROC |
|---|---|---|
| L0 | always predict "no opacity" | 0.5000 |
| L1 | logistic regression on 1,024 raw pixels | **0.7400** |
| L2 | logistic regression on 512 frozen ResNet-18 ImageNet features | 0.7250 |
| **L3** | **DenseNet121, pretrained, fine-tuned (this notebook)** | *measured below* |

**The number L3 has to beat is 0.740 on test AUROC.** If it does not, that is a real finding and we write it up
rather than tuning thresholds until it does.

Read `02_baseline_logreg.ipynb` first. The train/val/test split used here is *the same file* (`results/split_*.csv`),
so the numbers are comparable line for line.

### Plan

1. Get the images onto the machine (Kaggle: attach the competition; Colab: the zip already in Drive - see the data cell).
2. **Decode every DICOM once** into small JPEGs. This is the whole reason the run fits in an hour: decoding costs
   ~60 ms per file, and an epoch touches every file. Measured in notebook 02, not guessed.
3. DenseNet121 (ImageNet weights), head swapped to a single logit, `BCEWithLogitsLoss` with `pos_weight` set from
   the *train* prevalence (0.2253).
4. Two frozen epochs to warm the head, then unfreeze with a low backbone learning rate.
5. Pick the decision threshold on **validation** by Youden's J, then open the **test** split exactly once.
6. Write `densenet_metrics.json` in the same schema as `baseline_metrics.json`, plus the weights - to
   `/kaggle/working` on Kaggle (kept as the kernel's output) or to Drive on Colab.

> For education only. Not for clinical use. See `docs/model_card.md` for the limits and bias statement.

In [ ]:
import os, sys, time, json, platform
print("python", sys.version.split()[0], "|", platform.release())
try:
    import torch, torchvision
    print("torch", torch.__version__, "| torchvision", torchvision.__version__)
    print("cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("device:", torch.cuda.get_device_name(0),
              "| free VRAM:", round(torch.cuda.mem_get_info()[0] / 2**30, 1), "GB")
except ImportError as exc:
    print("install dependencies in the next cell:", exc)

In [ ]:
# Kaggle images ship pydicom and scikit-learn and have no internet by default; Colab needs the install.
# No %pip here: a line magic cannot sit inside a try block, and this must degrade quietly offline.
try:
    import pydicom, sklearn
    print("pydicom", pydicom.__version__, "| scikit-learn", sklearn.__version__, "- already in the image")
except ImportError:
    import subprocess
    print("installing pydicom + scikit-learn ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-warn-conflicts",
                    "pydicom", "scikit-learn"], check=True)

import torch
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| free VRAM:",
          round(torch.cuda.mem_get_info()[0] / 2**30, 1), "GB")
else:
    print("no GPU at import time - allowed only on a local CPU laptop; the config cell raises on Kaggle/Colab")

## 1. Get the data onto the machine

Three routes, and the notebook works out which one applies instead of assuming a path.

**Kaggle - attach the competition.** Add Input > Competitions > `rsna-pneumonia-detection-challenge`.
The input ships the images in one of two shapes - `stage_2_train_images.zip` (older layout) or the
already-extracted `stage_2_train_images/` folder holding the 26,684 `.dcm` files (current layout) - and
the cell below detects whichever is attached. Given the folder, it reads the pixels in place: no unzip,
no second 3.4 GB copy onto the ephemeral disk. The JPEG cache goes to `/tmp` because `/kaggle/working`
is what Kaggle saves as the kernel output.

**Colab - the original zip, already in Drive.** The same archive is at the top level of My Drive, which is
one file rather than 26,684 uploads. Colab's own Drive mount needs no third-party OAuth client, which is
exactly why this route works and why a plain `dvc push` from the laptop needed a registered client of its
own. The cell below searches a few likely folders for the zip instead of trusting one path.

**`dvc pull` - documented in `README.md` ("Versioning the data with DVC").** Two things make it awkward
inside a hosted VM: Google refuses the plugin's shared OAuth client, which is why a personal *Desktop app*
client had to be registered, and a browser OAuth loopback cannot run inside a Colab or Kaggle VM at all.
The image folder is also 26,688 separate cache objects, which is why the archive is tracked as a single DVC
object alongside them.


In [ ]:
import shutil

# One notebook, three hosts. The host is read off the filesystem rather than from a flag someone has to
# remember to flip, and RUN_ON is printed and stored in densenet_metrics.json's runtime block so a reviewer
# can see which machine produced the numbers. "local" means a CPU laptop: that run is a smoke test of the
# code, not a training run, and must not be recorded as though a T4 did it.
RUN_ON = "kaggle" if os.path.isdir("/kaggle") else ("colab" if os.path.isdir("/content") else "local")

# The GPU is mandatory on a hosted runtime: those are the numbers the report quotes, and a CPU fallback
# there would silently turn a 40-minute T4 run into an hours-long one. The laptop is the exception - a
# local pass exists to prove the code runs, and its metrics are never quoted.
if RUN_ON != "local" and not torch.cuda.is_available():
    raise AssertionError("no GPU attached. Colab: Runtime > Change runtime type > T4. "
                         "Kaggle: right sidebar > Accelerator > GPU (T4 or P100)")

# The repository folder, when this notebook is run from notebooks/ as intended.
_LOCAL_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
_LOCAL_EXTRACTED = os.path.join(_LOCAL_ROOT, "data", "raw", "rsna", "stage_2_train_images")

PATHS = {
    # Colab: scratch on the ephemeral local SSD, deliverables through the Drive mount.
    "colab": {"data": "/content/rsna", "cache": "/content/rsna_cache", "repo_src": "/content/repo_src",
              "drive_root": "/content/drive/MyDrive/aiml02",
              "results": "/content/drive/MyDrive/aiml02/results",
              "artifacts": "/content/drive/MyDrive/aiml02/artifacts"},
    # Kaggle: /kaggle/working becomes the kernel's saved output, so 3.4 GB of extracted pixels goes to
    # /tmp and only the metrics JSON and checkpoints are left in working.
    "kaggle": {"data": "/tmp/rsna", "cache": "/tmp/rsna_cache", "repo_src": "/tmp/repo_src",
               "drive_root": "/kaggle/working",
               "results": "/kaggle/working",
               "artifacts": "/kaggle/working"},
    # Local: a laptop. Scratch lives under notebooks/_nb03_scratch - including repo_src, because pointing
    # it at the working tree would unzip the code archive on top of the repository.
    "local": {"data": os.path.join(os.getcwd(), "_nb03_scratch", "rsna"),
              "cache": os.path.join(os.getcwd(), "_nb03_scratch", "rsna_cache"),
              "repo_src": os.path.join(os.getcwd(), "_nb03_scratch", "repo_src"),
              "drive_root": os.path.join(os.getcwd(), "_nb03_scratch"),
              "results": os.path.join(os.getcwd(), "_nb03_scratch", "results"),
              "artifacts": os.path.join(os.getcwd(), "_nb03_scratch", "artifacts")},
}[RUN_ON]

CONFIG = {
    "run_on": RUN_ON,
    "repo_url": "",   # e.g. "https://github.com/<you>/AIML-02-medical-imaging.git" once pushed
    "drive_root": PATHS["drive_root"],
    "zip_name": "stage_2_train_images.zip",
    # A laptop that already ran `dvc pull` has the pixels; do not ask for a second 3.4 GB copy.
    "local_images": (_LOCAL_EXTRACTED if RUN_ON == "local" and os.path.isdir(_LOCAL_EXTRACTED)
                     else os.path.join(PATHS["data"], "stage_2_train_images")),
    "cache_root": PATHS["cache"],       # local SSD, ephemeral by design
    "repo_results": PATHS["results"],
    "out_dir": PATHS["artifacts"],
    "seed": 7,
    "cache_size": 448,      # decoded once, resized large enough for Grad-CAM overlays in notebook 04
    "train_size": 224,      # what the network sees
    "batch_train": 32,
    "batch_eval": 64,
    "epochs_frozen": 2,
    "epochs_total": 10,
    "lr_head": 1e-3,
    "lr_backbone": 1e-4,
    "patience": 3,
}

# Nothing under /content/drive is created here: creating a folder inside the mountpoint before
# drive.mount() is what made the mount fail with "Mountpoint must not already contain files".
# Folder creation happens in the next cell, after the mount succeeded.


def free_gb(path):
    # shutil.disk_usage needs a path that exists, and the image folder is only created by the unzip cell;
    # an attached Kaggle input can also be missing entirely. Measure the nearest existing parent.
    probe = os.path.abspath(path)
    while not os.path.isdir(probe):
        up = os.path.dirname(probe)
        if up == probe:
            return None
        probe = up
    return round(shutil.disk_usage(probe).free / 2**30, 1)


print("running on:", RUN_ON,
      "| free GB where images go:", free_gb(PATHS["data"]),
      "| free GB where the cache goes:", free_gb(PATHS["cache"]))
print(json.dumps(CONFIG, indent=2))

In [ ]:
import glob


def _dcm_count(folder):
    # os.scandir, never a shell glob: 26,684 names through globbing takes minutes on this filesystem.
    return sum(1 for e in os.scandir(folder) if e.name.endswith(".dcm")) if os.path.isdir(folder) else 0


if RUN_ON == "colab":
    from google.colab import drive
    # A runtime that already mounted, or a cell that wrote into /content/drive first, leaves files in
    # the mountpoint and drive.mount refuses those. Remounting is safe: nothing of ours lives there yet.
    try:
        drive.mount("/content/drive", force_remount=False)
    except ValueError as exc:
        print("first mount attempt failed:", exc, "- forcing a remount")
        drive.mount("/content/drive", force_remount=True)
    assert os.listdir("/content/drive"), "Drive mounted but /content/drive is empty - re-run and authorise the OAuth dialog"
    SEARCH = [os.path.join(d, CONFIG["zip_name"]) for d in
              ("/content/drive/MyDrive", "/content/drive/MyDrive/aiml02",
               "/content/drive/MyDrive/Colab Notebooks")]
    EXTRACTED = []
elif RUN_ON == "kaggle":
    # Kaggle: Add Input > Competitions > rsna-pneumonia-detection-challenge. The archive is already on
    # the platform, so there is no 3.3 GB upload and no Drive mount at all. The input *used* to ship
    # stage_2_train_images.zip; the current one is the already-extracted stage_2_train_images/ folder,
    # so hunt for both shapes and take whichever is actually attached.
    SEARCH = sorted(glob.glob("/kaggle/input/**/" + CONFIG["zip_name"], recursive=True))
    EXTRACTED = sorted(p for p in glob.glob("/kaggle/input/**/stage_2_train_images", recursive=True)
                       if _dcm_count(p) >= 1000)
else:
    # Local: the `dvc pull` in the README leaves the archive in data/raw/rsna/.
    SEARCH = [os.path.join(_LOCAL_ROOT, "data", "raw", "rsna", CONFIG["zip_name"])]
    EXTRACTED = []
hits = [p for p in SEARCH if os.path.exists(p)]

zip_path = hits[0] if hits else None
if zip_path is None and EXTRACTED:
    # Current Kaggle layout: read the pixels where they sit. The input mount is read-only, which is
    # fine - nothing here writes to it; the JPEG cache and every artifact go to /tmp and /kaggle/working.
    CONFIG["local_images"] = EXTRACTED[0]
    print("no zip, but the extracted folder is attached:")
    print("  using", CONFIG["local_images"], f"({_dcm_count(CONFIG['local_images']):,} dcm) - unzip skipped")
if zip_path is None and not EXTRACTED:
    extracted = (sum(1 for _ in os.scandir(CONFIG["local_images"]))
                 if os.path.isdir(CONFIG["local_images"]) else 0)
    if not (RUN_ON == "local" and extracted >= 1000):
        attached = {}
        if RUN_ON == "kaggle" and os.path.isdir("/kaggle/input"):
            for src in sorted(os.listdir("/kaggle/input")):
                top = os.path.join("/kaggle/input", src)
                attached[src] = sorted(os.listdir(top))[:8] if os.path.isdir(top) else "(file)"
        hint = ("Kaggle: Add Input > Competition > rsna-pneumonia-detection-challenge, then re-run. "
                f"What is actually attached under /kaggle/input: {attached}"
                if RUN_ON == "kaggle" else
                "Colab: upload stage_2_train_images.zip (3.3 GB) to My Drive, then re-run.\n"
                "Local: run `dvc pull`, or point CONFIG[\"local_images\"] at the extracted folder - "
                f"{CONFIG['local_images']} holds {extracted} files.")
        raise FileNotFoundError(f"{CONFIG['zip_name']} not found. Looked in: {SEARCH}\n" + hint)
    print(f"no archive, but {extracted:,} files are already extracted at {CONFIG['local_images']} "
          "- the unzip cell uses them and skips a 45-second extract")
if zip_path is not None:
    print("using zip:", zip_path, f"({os.path.getsize(zip_path):,} bytes)")
    if len(hits) > 1:
        print("also present at:", hits[1:], "- deleting duplicates saves quota")

for d in (CONFIG["drive_root"], CONFIG["repo_results"], CONFIG["out_dir"]):
    os.makedirs(d, exist_ok=True)
print("output folders ready:", CONFIG["repo_results"], "|", CONFIG["out_dir"])


In [ ]:
import shutil, subprocess, zipfile

EXTRACT_TO = PATHS["repo_src"]
REPO = None
CODE_MARKER = os.path.join("src", "data", "dicom_io.py")

# Same policy as the archive search: look in several places instead of assuming where the code ended
# up, because re-zipping and re-uploading mid-run costs more than the check.
CANDIDATES = [
    "/content/repo.zip",                                   # dragged into the Colab Files panel
    "/kaggle/working/repo.zip",                            # uploaded into the Kaggle notebook session
    "/content/drive/MyDrive/repo.zip",                     # uploaded to My Drive root
    "/content/drive/MyDrive/aiml02/repo.zip",
    "/content/drive/MyDrive/Colab Notebooks/repo.zip",
]
# Kaggle's Drive integration (notebook Settings > Google Drive) mounts My Drive under /kaggle/input/drive,
# and an attached dataset lands anywhere under /kaggle/input - so glob both instead of fixing a path.
CANDIDATES += sorted(glob.glob("/kaggle/input/**/repo.zip", recursive=True))
# Kaggle unzips a zip uploaded to a dataset, so the attached code archive can also arrive as a plain tree
# at /kaggle/input/<dataset>/ instead of <dataset>/repo.zip. Look for the marker file that way too.
_MARKER_GLOB = CODE_MARKER.replace(os.sep, "/")
EXTRACTED_INPUT = []
for _dirpath, _dirnames, _files in os.walk("/kaggle/input"):
    if "competitions" in _dirnames:
        _dirnames.remove("competitions")   # 26k RSNA DICOMs live there and no code
    if os.path.exists(os.path.join(_dirpath, CODE_MARKER)):
        EXTRACTED_INPUT.append(_dirpath)
        _dirnames[:] = []
repo_zip = next((p for p in CANDIDATES if os.path.exists(p)), None)

if repo_zip:
    print("using code archive:", repo_zip, f"({os.path.getsize(repo_zip):,} bytes)")
    shutil.rmtree(EXTRACT_TO, ignore_errors=True)
    os.makedirs(EXTRACT_TO, exist_ok=True)
    with zipfile.ZipFile(repo_zip) as zf:
        zf.extractall(EXTRACT_TO)
elif EXTRACTED_INPUT:
    # /kaggle/input is read-only and the cells below write results next to the code they import, so the
    # attached tree is copied rather than imported in place.
    _src_dir = EXTRACTED_INPUT[0].replace("/" + _MARKER_GLOB, "")
    print("Kaggle unzipped the attached code archive - copying", _src_dir, "->", EXTRACT_TO)
    shutil.rmtree(EXTRACT_TO, ignore_errors=True)
    shutil.copytree(_src_dir, EXTRACT_TO)
elif RUN_ON == "local":
    # No archive on this laptop: use the repository this notebook is being edited in. The marker search
    # below walks from PATHS["repo_src"], which for a local run is the working tree itself.
    EXTRACT_TO = _LOCAL_ROOT
    print("local run - using the working tree as the code source:", EXTRACT_TO)
elif CONFIG["repo_url"]:
    shutil.rmtree(EXTRACT_TO, ignore_errors=True)
    subprocess.run(["git", "clone", "--depth", "1", CONFIG["repo_url"], EXTRACT_TO], check=True)
    print("cloned:", CONFIG["repo_url"])
else:
    raise FileNotFoundError(
        "The hosted notebook has no copy of the project code. Any one of these fixes it:\n"
        "  1. Colab: drag repo.zip into the Files panel, or upload it to My Drive/aiml02/;\n"
        "  2. Kaggle: Settings > Google Drive (then repo.zip is under /kaggle/input/drive), or attach\n"
        "     repo.zip as a dataset, or upload it in the notebook's Files sidebar;\n"
        "  3. set CONFIG['repo_url'] to the GitHub repository.\n"
        "repo.zip is the code-only archive (~1.9 MB) built on the laptop - see README."
    )

# The directory holding src/ is whichever directory *contains* the marker file. A zip stored with
# files at its top level extracts to EXTRACT_TO itself; a zip of a folder extracts one level deeper,
# so take the shallowest match rather than assuming either shape.
matches = [dp for dp, _dirs, _files in os.walk(EXTRACT_TO)
           if os.path.exists(os.path.join(dp, CODE_MARKER))]
if not matches:
    raise RuntimeError(
        f"no {CODE_MARKER} under {EXTRACT_TO}. Zip the repository folder itself (the one holding "
        "src/, tests/, configs/), not loose files."
    )
REPO = min(matches, key=lambda p: p.count(os.sep))
sys.path.insert(0, REPO)
print("repo at:", REPO)

from src.data.dicom_io import to_uint8

print("windowing imported from src/data/dicom_io.py - the same code tests/test_dicom_io.py covers")

# The pretrained backbone needs the ImageNet weights file to travel inside repo.zip: a Kaggle kernel
# has no internet by default, and without the file torchvision silently tries to download the weights
# and dies on a network timeout three cells from here. Fail at the door with the fix instead.
from src.models.classifier import WEIGHTS_FILENAME
weights_path = os.path.join(REPO, "weights", WEIGHTS_FILENAME)
if not os.path.exists(weights_path):
    raise FileNotFoundError(
        f"{weights_path} is missing from the code archive - the attached repo.zip is the pre-weights "
        "build. Fix: on Kaggle, open your private repo dataset > New Version > Drag the fresh repo.zip "
        "into it (the ~38 MB one that sits next to the repository checkout on the laptop), update the "
        "notebook's input to that version, re-run.")
print("ImageNet weights present in the archive:", weights_path, f"({os.path.getsize(weights_path):,} bytes)")

# The split CSVs and the baseline bar come from git, so check them here rather than three cells later.
for need in (os.path.join("results", f"split_{s}.csv") for s in ("train", "val", "test")):
    assert os.path.exists(os.path.join(REPO, need)), f"{need} is not in the code archive"
print("split CSVs present in", os.path.join(REPO, "results"))

In [ ]:
BASE = os.path.dirname(CONFIG["local_images"])


def dcm_count(folder):
    # os.scandir, never a shell glob: 26,684 names through globbing on this filesystem takes minutes.
    return sum(1 for e in os.scandir(folder) if e.name.endswith(".dcm")) if os.path.isdir(folder) else 0


# The archive stores 26,684 loose .dcm files at its root - there is no stage_2_train_images/ folder inside it,
# so extracting into BASE lands the images directly in BASE. On the laptop the folder exists only because
# README's `unzip -d stage_2_train_images` created it by hand, which is what hid this assumption. Detect the
# layout before unzipping, because a second 3.3 GB extract is 45 s of Colab time and a second chance to fail.
if dcm_count(CONFIG["local_images"]) < 1000 and dcm_count(BASE) < 1000:
    if zip_path is None or not os.path.exists(zip_path):
        raise FileNotFoundError(
            f"{CONFIG['zip_name']} is not on disk and {CONFIG['local_images']} holds fewer than 1,000 "
            "images. Colab: upload stage_2_train_images.zip (3.3 GB) to MyDrive/aiml02/. Kaggle: Add Input "
            "> Competition > rsna-pneumonia-detection-challenge. Local: `dvc pull` (README)."
        )
    t0 = time.time()
    os.makedirs(BASE, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(BASE)
    print(f"unzipped in {time.time() - t0:.0f} s")

if dcm_count(CONFIG["local_images"]) < 1000:
    for root, _dirs, _files in os.walk(BASE):        # covers loose-at-root and any nested folder name
        if dcm_count(root) >= 1000:
            CONFIG["local_images"] = root
            break

print("images at:", CONFIG["local_images"])
n_dcm = dcm_count(CONFIG["local_images"])
print("DICOM files on disk:", n_dcm, "(expected 26684)")
assert n_dcm == 26684, "image count does not match the DVC md5 / eda_summary.json figure"

In [ ]:
import pandas as pd

splits = {}
for name in ("train", "val", "test"):
    csv = os.path.join(REPO, "results", f"split_{name}.csv")
    if not os.path.exists(csv):
        raise FileNotFoundError(
            f"{csv} is missing from the checkout. These three CSVs are committed to git on purpose: they are "
            "what makes every notebook, the API and the fairness audit use one identical split."
        )
    splits[name] = pd.read_csv(csv)
    print(name, len(splits[name]), "| opacity rate", round(splits[name]["label"].mean(), 4))

# Same assertion the baselines made: one image per row, no row-level duplicates.
for name, frame in splits.items():
    assert frame["dicom_rel"].is_unique, f"{name}: duplicate dicom_rel"
    assert not set(frame["patient_id"]) & set(
        pd.concat([f for n, f in splits.items() if n != name])["patient_id"]
    ), f"{name}: patient leaked across splits"
print("no patient overlap between splits - verified on the files actually loaded")

## 2. Decode once: DICOM -> windowed 8-bit -> 448 px JPEG

`src/data/dicom_io.py:85-88` carries the bug that cost a day of debugging: these RSNA files are **8-bit Secondary
Captures** (verified: `BitsAllocated == 8` for all 26,684 files, `WindowCenter` absent for all of them). Applying the
default Hounsfield window to an already display-encoded image squeezes the chest into a handful of grey levels. The
function imported above is the corrected one and `tests/test_dicom_io.py` pins that behaviour, so the hosted run and
the laptop tests cannot drift apart. They are stored at 448 px rather than at network input size so that notebook 04's
Grad-CAM overlays have something to be drawn on - and a laptop run that keeps its cache finds these JPEGs already
there, while Kaggle wipes `/tmp` between kernels and notebook 04 rebuilds only the images it needs.

Four worker processes: decode is CPU-bound and both Kaggle and Colab hand over four cores.

In [ ]:
import numpy as np
from concurrent.futures import ProcessPoolExecutor
from PIL import Image


def _render(task):
    import pydicom
    src, dst, size = task
    if os.path.exists(dst):
        return "skip"
    try:
        img = Image.fromarray(to_uint8(pydicom.dcmread(src, force=True)))
    except Exception as exc:  # a single unreadable file must not lose a 40-minute run
        return f"error {os.path.basename(src)}: {type(exc).__name__}: {exc}"
    img = img.resize((size, size), Image.BILINEAR).convert("L")
    img.save(dst, "JPEG", quality=90)
    return "ok"


def build_cache(split: str) -> str:
    out = os.path.join(CONFIG["cache_root"], split)
    os.makedirs(out, exist_ok=True)
    tasks = [(os.path.join(CONFIG["local_images"], os.path.basename(r)),
              os.path.join(out, os.path.basename(r).replace(".dcm", ".jpg")),
              CONFIG["cache_size"])
             for r in splits[split]["dicom_rel"]]
    t0, results = time.time(), []
    def _drain(iterator):
        for i, res in enumerate(iterator, 1):
            results.append(res)
            if i % 4000 == 0:
                el = time.time() - t0
                print(f"  {split}: {i}/{len(tasks)} ({el:.0f} s, {el/i*1000:.0f} ms/file, "
                      f"eta {(len(tasks)-i)*el/i/60:.1f} min)", flush=True)


    if os.name == "nt":
        # Windows spawns workers and cannot re-import a function defined in a notebook cell, so the
        # pool fails before decoding anything. The laptop path runs the same loop serially instead.
        _drain(map(_render, tasks))
    else:
        with ProcessPoolExecutor(max_workers=4) as pool:
            _drain(pool.map(_render, tasks, chunksize=32))
    bad = [r for r in results if r != "ok" and r != "skip"]
    print(f"{split}: {len(results)-len(bad)} rendered, {len(bad)} failed in {time.time()-t0:.0f} s")
    if bad:
        print("  first failures:", bad[:5])
    return out


if not os.path.isdir(os.path.join(CONFIG["cache_root"], "train")):
    for split in ("train", "val", "test"):
        build_cache(split)
else:
    print("cache already present - skipping decode (delete /content/rsna_cache to rebuild)")

for split in ("train", "val", "test"):
    n = len(os.listdir(os.path.join(CONFIG["cache_root"], split)))
    print(split, n, "jpg |", len(splits[split]), "rows")
    assert n >= 0.995 * len(splits[split]), f"{split}: cache coverage below 99.5% of the split"

In [ ]:
# Size and prevalence sanity check straight off the cache, not off the documentation.
from sklearn.utils.class_weight import compute_sample_weight

prevalence = float(splits["train"]["label"].mean())
pos_weight = torch.tensor((1 - prevalence) / prevalence, dtype=torch.float32)
print(f"train prevalence {prevalence:.4f} -> pos_weight {pos_weight.item():.3f}")
sample = Image.open(os.path.join(CONFIG["cache_root"], "train",
                                 os.listdir(os.path.join(CONFIG["cache_root"], "train"))[0]))
print("cache image:", sample.size, sample.mode, "| bytes",
      os.path.getsize(os.path.join(CONFIG["cache_root"], "train", os.listdir(CONFIG["cache_root"] + "/train")[0])))
# os.walk, not 'du': there is no du on a Windows laptop, and the shell error it printed looked like a
# cell failure. One pass over the cache files is a second even on the full 26,684-image Kaggle cache.
cache_bytes = sum(os.path.getsize(os.path.join(d, f))
                  for d, _, fs in os.walk(CONFIG["cache_root"]) for f in fs)
print(f"cache on disk: {cache_bytes / 2**30:.2f} GB ({cache_bytes / 2**20:.0f} MB)")

## 3. Dataset and model

Torchvision has changed the shape of the `densenet121` head between releases: through 0.19 it is
`nn.Sequential(Linear(1024, 1000), LogSoftmax(dim=1))`, and from 0.20 it is a bare `nn.Linear(1024, 1000)`. Colab
reports `torchvision 0.26.0+cu130`, so the bare form is what runs here - but the cell below reads the width off
whichever exists rather than hard-coding `.classifier.linear.in_features`, which died on the first Colab attempt with
`AttributeError: 'Linear' object has no attribute 'linear'`.

Either way the **whole** `classifier` is replaced with one `nn.Linear(1024, 1)`, never just the inner layer: on the
Sequential form, leaving `LogSoftmax` in the graph on a single logit returns 0.0 for every input, so the model appears
to train while sitting at AUROC 0.5 forever. The probe asserts two distinct logits, so that trap - and any future head
change - fails loudly instead of silently. The probe input is `randn`, not zeros: two identical rows would produce two
identical outputs no matter what the head looks like, and the assertion would be measuring nothing.

Images are normalised with ImageNet statistics: the backbone is ImageNet-pretrained, so keeping its input
distribution is the cheapest way to make two frozen epochs useful.

In [ ]:
from torch.utils.data import DataLoader

from src.data.chest_dataset import ChestXrayDataset
from src.data.transforms import make_transform


def loader(split, train=False):
    ds = ChestXrayDataset(splits[split], os.path.join(CONFIG["cache_root"], split),
                          make_transform(CONFIG["train_size"], train), CONFIG["cache_size"])
    return DataLoader(ds, batch_size=CONFIG["batch_train"] if train else CONFIG["batch_eval"],
                      shuffle=train, num_workers=2, pin_memory=True, drop_last=train), len(ds)


for name, frame in splits.items():
    _l, n = loader(name)
    print(f"{name}: {n} images, {int(frame['label'].sum())} positive ({frame['label'].mean():.4f})")
    print(f"  rows with no render in the cache: {len(_l.dataset.missing_paths())}")

In [ ]:
import torch.nn as nn

from src.models.classifier import build_densenet121, set_frozen

# The head swap and its probe live in src/models/classifier.py, where the torchvision <=0.19 vs
# >=0.20 classifier difference is handled and tested; build_densenet121 asserts distinct finite
# logits before it returns, so a surviving LogSoftmax cannot reach the training loop.
net = build_densenet121(num_classes=1, repo_dir=REPO, probe_size=CONFIG["train_size"])
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu" and CONFIG["run_on"] != "local":
    raise AssertionError("no GPU on a hosted host: Kaggle sidebar > Accelerator > GPU T4, or "
                         "Colab Runtime > Change runtime type > T4")
net.to(device)
print("head:", net.classifier)
print("params:", f"{sum(p.numel() for p in net.parameters()):,}")

## 4. Training loop

Two frozen epochs warm the 1,025-parameter head, then the backbone joins at `lr_backbone` (one tenth of the head's,
so 1.2 M of pretrained features are not washed out by 18,678 noisy labels). Mixed precision on a T4 roughly doubles
throughput. Early stopping watches **validation AUROC**, and the best state dict is kept in RAM.

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

torch.manual_seed(CONFIG["seed"]); np.random.seed(CONFIG["seed"])
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True   # faster, but makes runs not bit-identical; noted in the model card


def metrics_from(probs, y, thr):
    pred = (probs >= thr).astype(int)
    tp = int(((pred == 1) & (y == 1)).sum()); fp = int(((pred == 1) & (y == 0)).sum())
    fn = int(((pred == 0) & (y == 1)).sum()); tn = int(((pred == 0) & (y == 0)).sum())
    sens = tp / (tp + fn) if tp + fn else 0.0
    spec = tn / (tn + fp) if tn + fp else 0.0
    prec = tp / (tp + fp) if tp + fp else 0.0
    f1 = 2 * prec * sens / (prec + sens) if prec + sens else 0.0
    return {
        "n": int(len(y)),
        "auroc": round(float(roc_auc_score(y, probs)), 4),
        "pr_auc": round(float(average_precision_score(y, probs)), 4),
        "threshold": round(float(thr), 4),
        "accuracy": round((tp + tn) / len(y), 4),
        "sensitivity": round(sens, 4), "specificity": round(spec, 4),
        "precision": round(prec, 4), "f1": round(f1, 4),
        "confusion": {"tn": tn, "fp": fp, "fn": fn, "tp": tp},
    }


@torch.no_grad()
def predict(net, split, limit=None):
    net.eval()
    ldr, n = loader(split)
    if limit:
        from torch.utils.data import Subset
        idx = np.linspace(0, n - 1, min(limit, n)).astype(int)
        ldr = DataLoader(Subset(ldr.dataset, idx), batch_size=CONFIG["batch_eval"],
                         shuffle=False, num_workers=2)
        n = len(idx)
    probs, ys = [], []
    for x, y in ldr:
        with torch.amp.autocast("cuda", dtype=torch.float16, enabled=(device == "cuda")):
            logits = net(x.to(device, non_blocking=True)).squeeze(1)
        probs.append(torch.sigmoid(logits.float()).cpu().numpy())
        ys.append(y.numpy())
    return np.concatenate(probs), np.concatenate(ys)


def youden_threshold(probs, y):
    """Youden's J on the validation split. Always returns a finite threshold in [0, 1]."""
    from sklearn.metrics import roc_curve
    fpr, tpr, thr = roc_curve(y, probs)
    if len(thr) == 0:
        return 0.5
    # roc_curve puts +/-inf in the corner rows ("classify everything one way / the other"). If the best J
    # lands on an inf row the predictions are effectively constant - epoch 0 of a frozen backbone does that -
    # and a threshold of inf would silently classify every image negative and be written into the JSON
    # payload as the non-standard token `Infinity`. Fall back to the middle of the finite thresholds.
    k = int(np.argmax(tpr - fpr))
    t = float(thr[k])
    if not np.isfinite(t):
        finite = thr[np.isfinite(thr)]
        t = float(np.median(finite)) if len(finite) else 0.5
    return min(max(t, 0.0), 1.0)

In [ ]:
from src.models.classifier import save_checkpoint


def train():
    net = build_densenet121(num_classes=1, repo_dir=REPO, probe_size=CONFIG["train_size"]).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))
    head, backbone = [], []
    for name, p in net.named_parameters():
        (backbone if name.startswith("features.") else head).append(p)
    scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))
    history, best = [], {"auroc": 0.0, "state": None, "epoch": -1}
    t_start = time.time()

    for epoch in range(CONFIG["epochs_total"]):
        frozen = epoch < CONFIG["epochs_frozen"]
        set_frozen(net, frozen)
        optimizer = torch.optim.AdamW(
            [{"params": backbone, "lr": CONFIG["lr_backbone"] if not frozen else 0.0},
             {"params": head, "lr": CONFIG["lr_head"]}],
            weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG["epochs_total"] - epoch)
        ldr, n = loader("train", train=True)
        net.train(); running = 0.0
        for x, y in ldr:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", dtype=torch.float16, enabled=(device == "cuda")):
                loss = criterion(net(x).squeeze(1), y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(backbone + head, 5.0)
            scaler.step(optimizer); scaler.update()
            running += float(loss.detach()) * len(y)   # detach: float() on a grad tensor warns and syncs unnecessarily
        sched.step()
        vp, vy = predict(net, "val")
        thr = youden_threshold(vp, vy)
        row = {"epoch": epoch, "frozen": frozen, "train_loss": round(running / n, 4),
               "val_auroc": round(float(roc_auc_score(vy, vp)), 4),
               "val_pr_auc": round(float(average_precision_score(vy, vp)), 4),
               "val_threshold": round(thr, 4), "minutes": round((time.time() - t_start) / 60, 1)}
        history.append(row)
        print(json.dumps(row))
        # Save every epoch, not just at the end. A free-tier session can be disconnected halfway
        # through a 2-hour run, and losing 8 epochs to a timeout is the difference between finishing
        # this week and not. ~28 MB to the host disk costs seconds next to a 10-minute epoch.
        save_checkpoint(net, os.path.join(CONFIG["out_dir"], "densenet_last.pt"), CONFIG,
                        epoch=epoch, history=history,
                        extra={"note": "resume point, overwritten every epoch; the graded artefact is cell 21's"})
        if row["val_auroc"] > best["auroc"]:
            best = {"auroc": row["val_auroc"], "state": {k: v.detach().cpu().clone()
                                                         for k, v in net.state_dict().items()}, "epoch": epoch}
        if epoch >= CONFIG["epochs_frozen"] and epoch - best["epoch"] >= CONFIG["patience"]:
            print(f"early stop: no val AUROC gain for {CONFIG['patience']} epochs")
            break

    net.load_state_dict(best["state"])
    print(f"best epoch {best['epoch']} with val AUROC {best['auroc']:.4f} "
          f"| total {(time.time()-t_start)/60:.1f} min")
    return net, history, best


model, history, best = train()
torch.cuda.empty_cache()

## 5. Threshold on validation, then open the test split once

The threshold is chosen by Youden's J on **validation** only. Once `test_metrics` is computed the split is closed -
re-running that cell on a new threshold would silently turn the test number into a validation number. If you need to
change something, go back and retrain; do not re-cut the test set.

`n = 4003` images, prevalence 0.2253, identical to notebook 02, so the table below is directly comparable.

In [ ]:
train_p, train_y = predict(model, "train", limit=2000)
val_p, val_y = predict(model, "val")
test_p, test_y = predict(model, "test")
threshold = youden_threshold(val_p, val_y)

runs = []
for tag, (p, y) in {"L3 DenseNet121 - TRAIN (subset of 2,000)": (train_p, train_y),
                   "L3 DenseNet121 - VAL": (val_p, val_y)}.items():
    m = metrics_from(p, y, 0.5 if "TRAIN" in tag else threshold); m["model"] = tag; runs.append(m)
test_metrics = metrics_from(test_p, test_y, threshold); test_metrics["model"] = "L3 DenseNet121 - TEST"
val_metrics = runs[-1]

print(f"threshold from val Youden's J = {threshold:.4f} (notebook 02 chose 0.1628 for L1)")
print(f"train (subset) {runs[0]['auroc']:.4f} -> val {val_metrics['auroc']:.4f} | overfit gap "
      f"{runs[0]['auroc'] - val_metrics['auroc']:+.4f}  (L1's gap was +0.1517)")
print(json.dumps(test_metrics, indent=1))

bar = 0.7400
verdict = "BEATS" if test_metrics["auroc"] > bar else "DOES NOT BEAT"
print(f"\nL3 test AUROC {test_metrics['auroc']:.4f} {verdict} the L1 baseline bar of {bar}.")

In [ ]:
import matplotlib.pyplot as plt
# Deliberately NOT calling matplotlib.use("agg"): Colab renders through its inline backend, and agg
# turns plt.show() into a no-op, leaving the executed cell with no figure in it. The PNG is saved
# before show() instead, which is backend-independent.
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay

fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
RocCurveDisplay.from_predictions(test_y, test_p, ax=axes[0])
axes[0].plot([0, 1], [0, 1], "k--", lw=1); axes[0].set_title(f"ROC, test AUROC {test_metrics['auroc']:.3f}")
PrecisionRecallDisplay.from_predictions(test_y, test_p, ax=axes[1])
axes[1].axhline(float(test_y.mean()), color="k", ls="--", lw=1,
                label=f"chance = prevalence {test_y.mean():.3f}")
axes[1].legend(); axes[1].set_title(f"PR, AP {test_metrics['pr_auc']:.3f}")
axes[2].plot([h["epoch"] for h in history], [h["val_auroc"] for h in history], "o-", label="val AUROC")
axes[2].plot([h["epoch"] for h in history], [h["train_loss"] for h in history], "s--", label="train loss")
axes[2].axvline(CONFIG["epochs_frozen"] - 1, color="grey", ls=":", label="backbone unfrozen")
axes[2].set_xlabel("epoch"); axes[2].legend(); axes[2].set_title("training curve")
fig.suptitle("L3 DenseNet121 - for education only, not for clinical use", y=1.02)
fig.tight_layout()

# Write the file BEFORE showing the figure: plt.show() closes and clears it, and a savefig after
# that produces a blank PNG in the artifacts folder.
png_path = os.path.join(CONFIG["out_dir"], "densenet_training.png")
fig.savefig(png_path, dpi=130, bbox_inches="tight")
print("saved", png_path, f"({os.path.getsize(png_path):,} bytes)")
plt.show()

In [ ]:
from src.models.classifier import save_checkpoint

payload = {
    "generated": time.strftime("%Y-%m-%d %H:%M"),
    "dataset": "RSNA 2018 Pneumonia Detection Challenge, stage_2 release (label = lung opacity)",
    "model": "torchvision densenet121, IMAGENET1K_V1, head replaced by Linear(1024,1)",
    # which host actually produced this: Kaggle and Colab are both supported and the difference matters
    "runtime": {"host": CONFIG["run_on"], "device": str(device), "torch": torch.__version__,
                "cuda": torch.version.cuda, "seed": CONFIG["seed"],
                "python": platform.python_version()},
    "prevalence_train": prevalence,
    "pos_weight": round(float(pos_weight), 4),
    "threshold_by_youden_on_val": round(threshold, 4),
    "bar_to_beat_l1_pixel_baseline_test_auroc": bar,
    "verdict": f"{'beats' if test_metrics['auroc'] > bar else 'does not beat'} the L1 baseline",
    "epochs_run": len(history),
    "best_epoch": best["epoch"],
    "history": history,
    "evaluations": {r["model"]: r for r in runs + [test_metrics]},
    "cache": {"images_decoded_from": "DICOM -> windowed uint8 -> 448px JPEG q90",
              "windowing_rule": "src/data/dicom_io.py:to_uint8, 8-bit secondary-capture branch",
              "missing_files_treated_as_blank": int(sum(
                  1 for s in ("train", "val", "test")
                  for r in splits[s]["dicom_rel"]
                  if not os.path.exists(os.path.join(CONFIG["cache_root"], s,
                                                     os.path.basename(r).replace(".dcm", ".jpg")))))},
    "disclaimer": "For education only. Not for clinical use.",
}
out_json = os.path.join(CONFIG["repo_results"], "densenet_metrics.json")
with open(out_json, "w", encoding="utf-8") as fh:
    json.dump(payload, fh, indent=1)
save_checkpoint(model, os.path.join(CONFIG["out_dir"], "densenet121_lungopacity.pt"), CONFIG,
                epoch=best["epoch"], history=history,
                extra={"threshold": threshold, "class_names": ["No Lung Opacity", "Lung Opacity"],
                       "note": "graded artefact"})
print("wrote", out_json)
print("copy these two files back into the repo before the next session:")
print("  results/densenet_metrics.json")
print("  artifacts/densenet121_lungopacity.pt   (git-ignored; note its path in docs/model_card.md)")

## 6. Reading the result honestly

Check these five things before you believe the test number:

1. **Train-vs-val gap.** L1's gap was +0.1517 (0.9037 train, 0.7520 val). If L3's gap is similar or larger, the
   network is memorising, and the val AUROC - not the test one - is the number to quote in the presentation.
2. **PR-AUC chance level is the prevalence (0.2253), not 0.5.** On this label mix an AP of 0.45 is a genuine lift.
3. **Sensitivity and specificity at the chosen threshold.** A pneumonia surrogate where a human would want
   sensitivity above 0.8 needs explaining, not averaging away. Notebook 04 does the subgroup version of this.
4. **The 16.8 % row-level leak in the raw labels** (30,227 rows vs 26,684 images, and RSNA's de-identification gives
   every file a unique PatientID, so a patient-level split is impossible here). Same caveat as notebook 02, same
   effect on every number in this file - write it in the model card, do not pretend the split is clean.
5. **One test open.** If this cell is re-run after changing anything, the test number stops being a test number.

### Expected outcomes, stated before the run

- A fine-tuned DenseNet121 on RSNA lung opacity in the literature sits around 0.75-0.85 AUROC for the 2018 challenge
  test set. Anything in **0.76-0.82** on our val split is a healthy result.
- Below **0.75** on val is not a failure to hide - it says the 22.5 %-prevalence, single-view, 8-bit archive is
  harder than the published multi-view numbers, and that is a *finding* for the bias section.
- Above 0.90 would make me suspicious: check the split CSVs were really the repo's, and check the cache did not
  accidentally include a per-file label in its name.

### Troubleshooting

| symptom | cause / fix |
|---|---|
| `CUDA out of memory` | `CONFIG["batch_train"] = 16`, or `CONFIG["train_size"] = 192` and rebuild the cache |
| val AUROC stuck at 0.5 with zero loss change | the `LogSoftmax` trap - re-run the build cell and confirm the probe printed two distinct logits |
| `AttributeError: 'Linear' object has no attribute 'linear'` | a pre-0.20 assumption about the head; `head_in_features()` above reads whichever form torchvision installed |
| decode cell slower than ~25 min | you are writing the cache to Drive instead of `/content` (local SSD) |
| `zipfile.BadZipFile` | the Drive upload was interrupted; re-upload the 3.3 GB zip |
| session restarted mid-training | the cache and `/content` are ephemeral; expect to re-render (~25 min) then re-train |

### Next

`notebooks/04_gradcam_fairness.ipynb` takes the weights saved above and answers the two questions this notebook
cannot: *where* is it looking, and *for whom* does it work worse.